In [ ]:
import time
import requests
import warnings
from   pathlib import Path
warnings.filterwarnings("ignore")

API_URL         = "https://docslook.interrao.ru"
API_UPLOAD_URL  = f"{API_URL}/api/upload"
API_KEY         = "key1"
BATCH_SIZE      = 100

### Каталог с документами для распознования

In [ ]:
PATHDOCS        = r"R:\БВА_Проект_BI\Решения\Андрианов\1С_ГК_ПО_МЭС_ПУД\Архив_ПУД\АктПриемкиВыполненныхРаботОказанныхУслугПрисоединенныеФайлы"
files_to_upload = list( Path(PATHDOCS).rglob('*.pdf') )
print(files_to_upload)

### Основной процесс распознования 
Результат:
- [\\\\\msk1-vps02\\SHARE\\_OCR_ORIGIN](file://msk1-vps02/SHARE/_OCR_ORIGIN)
- [\\\\\msk1-vps02\\SHARE\\_OCR_TEXT](file://msk1-vps02/SHARE/_OCR_TEXT)

In [ ]:
# Формируем кортежи (имя параметра, (имя файла, бинарные данные, mime-type))
total = len(files_to_upload)
for i in range(0, total, BATCH_SIZE):
    print(f"эпоха {i}:{i+BATCH_SIZE}")
    job_id = None
    files  = [("files", (f.name, open(f, "rb"), "application/octet-stream")) for f in files_to_upload[i:i+BATCH_SIZE] ]
    headers= {"X-API-Key": API_KEY}
    data     = {"owner":"skip",
                "owner_email":"skip@local"}
    response = requests.post(API_UPLOAD_URL, 
                             headers=headers,
                             files=files, 
                             data=data,
                             verify=False)

    if response.ok:
        data   = response.json()
        job_id = data["job_id"]
        print("Job ID:", f'{API_URL}/jobs/{job_id}')
        print("Queued:", data["queued"])
        print("Path origin:",  f'file://msk1-vps02/SHARE/_OCR_ORIGIN/{data["prefix"]}')
        print("Path text:",  f'file://msk1-vps02/SHARE/_OCR_TEXT/{data["prefix"]}')
        print(f"Uploaded, job_id: {job_id}")
    else:
        print("Ошибка:", response.status_code, response.text)
    
    while True:
        time.sleep(3)
        status_resp = requests.get(f"{API_URL}/api/jobs/{job_id}", headers=headers,verify=False)
        status      = status_resp.json()
        
        print(f"Status: {status['status']}, Progress: {status['progress']}%")
        
        if status["status"] in ["completed", "unknown", "error"]:
            break